# 🚢 Survival Classification Estimator — End-to-End ML Pipeline

**Author:** Akarsh Shrivastav (24CSE1002)  
**Supervisor:** Dr. Keshavamurthy B.N  
**Course:** Undergraduate Seminar — Machine Learning Implementation  

---

## Pipeline Overview

| Stage | Description |
|-------|-------------|
| **1. Data Loading** | Load the Titanic Survival dataset via `seaborn` |
| **2. EDA** | Correlations, distributions, survival breakdowns |
| **3. Preprocessing** | `ColumnTransformer` with imputation, scaling, encoding |
| **4. Baseline Models** | Logistic Regression vs. Decision Tree (10-fold CV) |
| **5. Hyperparameter Tuning** | `GridSearchCV` on `RandomForestClassifier` |
| **6. Final Evaluation** | Evaluate best model on the untouched test set |
| **7. Model Persistence** | Serialize with `joblib` |

---
## 1 · Imports & Configuration

In [ ]:
# ── Standard Library ──────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# ── Data Handling ─────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-Learn ──────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
)

# ── Model Persistence ─────────────────────────────────────
import joblib

# ── Plot Aesthetics ───────────────────────────────────────
sns.set_theme(style="darkgrid", palette="viridis", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

RANDOM_STATE = 42
print("✅ All imports successful.")

---
## 2 · Data Loading & Initial Inspection

In [ ]:
# Load the Titanic dataset from seaborn's built-in datasets
df = sns.load_dataset("titanic")

print(f"Dataset shape: {df.shape}")
print(f"Survival distribution:\n{df['survived'].value_counts(normalize=True).round(3)}")
print()
df.head(10)

In [ ]:
# Data types and missing values
print("── Data Types ──")
print(df.dtypes)
print("\n── Missing Values ──")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nTotal missing cells: {df.isnull().sum().sum()} / {df.size} ({df.isnull().sum().sum() / df.size * 100:.2f}%)")

---
## 3 · Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Survival by Passenger Class ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Survival counts by class
sns.countplot(data=df, x="pclass", hue="survived", ax=axes[0])
axes[0].set_title("Survival by Passenger Class")
axes[0].set_xlabel("Class")
axes[0].legend(["Died", "Survived"])

# (b) Survival by sex
sns.countplot(data=df, x="sex", hue="survived", ax=axes[1])
axes[1].set_title("Survival by Sex")
axes[1].legend(["Died", "Survived"])

# (c) Age distribution by survival
sns.histplot(data=df, x="age", hue="survived", kde=True, bins=30, ax=axes[2])
axes[2].set_title("Age Distribution by Survival")
axes[2].legend(["Died", "Survived"])

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.2 Correlation Heatmap (numeric features) ───────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    square=True,
)
plt.title("Correlation Heatmap — Numeric Features")
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Fare distribution (log-scaled) by class ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="pclass", y="fare", hue="survived", ax=axes[0])
axes[0].set_title("Fare by Class & Survival")
axes[0].set_ylim(0, 300)

sns.countplot(data=df, x="embarked", hue="survived", ax=axes[1])
axes[1].set_title("Survival by Embarkation Port")
axes[1].legend(["Died", "Survived"])

plt.tight_layout()
plt.show()

---
## 4 · Feature Selection & Train-Test Split

In [ ]:
# ── Select features ──────────────────────────────────────
# We use a curated subset that balances signal with simplicity.
# Columns like 'deck' (77% missing) and 'embarked_town' (redundant) are dropped.

FEATURE_COLS = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "alone"]
TARGET_COL   = "survived"

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print(f"Feature matrix shape : {X.shape}")
print(f"Target vector shape  : {y.shape}")
print(f"Class balance        : {y.value_counts().to_dict()}")

In [ ]:
# ── Stratified Train-Test Split ───────────────────────────
# Stratification preserves the survival ratio in both partitions,
# which is critical for imbalanced datasets.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train set: {X_train.shape[0]} samples  |  Test set: {X_test.shape[0]} samples")
print(f"Train survival rate: {y_train.mean():.3f}")
print(f"Test  survival rate: {y_test.mean():.3f}")

---
## 5 · Preprocessing Pipeline (`ColumnTransformer`)

In [ ]:
# ── Define column groups ──────────────────────────────────
numeric_features     = ["age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "embarked", "pclass", "alone"]

# ── Numeric sub-pipeline ─────────────────────────────────
# 1. Impute missing values with the MEDIAN (robust to outliers)
# 2. Standardize to zero-mean, unit-variance
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# ── Categorical sub-pipeline ─────────────────────────────
# 1. Impute missing categories with the MODE (most frequent)
# 2. One-hot encode — drop first category to avoid collinearity
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
])

# ── Combined ColumnTransformer ────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer,     numeric_features),
        ("cat", categorical_transformer,  categorical_features),
    ],
    remainder="drop",  # Drop any columns not listed above
)

print("✅ Preprocessing pipeline configured.")
print(f"   Numeric features     : {numeric_features}")
print(f"   Categorical features : {categorical_features}")

---
## 6 · Baseline Model Comparison (10-Fold Stratified CV)

In [ ]:
# ── Define the cross-validation strategy ─────────────────
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# ── Model A: Logistic Regression ─────────────────────────
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

lr_cv_scores = cross_val_score(
    lr_pipeline, X_train, y_train,
    cv=cv_strategy, scoring="accuracy", n_jobs=-1,
)

# ── Model B: Decision Tree ───────────────────────────────
dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

dt_cv_scores = cross_val_score(
    dt_pipeline, X_train, y_train,
    cv=cv_strategy, scoring="accuracy", n_jobs=-1,
)

# ── Print results ─────────────────────────────────────────
print("═" * 55)
print(" 10-FOLD STRATIFIED CROSS-VALIDATION RESULTS")
print("═" * 55)
print(f"  Logistic Regression : {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")
print(f"  Decision Tree       : {dt_cv_scores.mean():.4f} ± {dt_cv_scores.std():.4f}")
print("═" * 55)

In [ ]:
# ── Visualize CV score distributions ─────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

cv_data = pd.DataFrame({
    "Logistic Regression": lr_cv_scores,
    "Decision Tree":       dt_cv_scores,
})

sns.boxplot(data=cv_data, palette="Set2", ax=ax)
ax.set_ylabel("Accuracy")
ax.set_title("10-Fold CV Accuracy — Baseline Comparison")
ax.set_ylim(0.6, 1.0)

# Add individual fold points
for i, col in enumerate(cv_data.columns):
    ax.scatter(
        np.full(10, i) + np.random.normal(0, 0.02, 10),
        cv_data[col], alpha=0.6, s=30, color="black", zorder=5,
    )

plt.tight_layout()
plt.show()

---
## 7 · Hyperparameter Tuning — `GridSearchCV` on Random Forest

In [ ]:
# ── Random Forest pipeline ───────────────────────────────
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   RandomForestClassifier(random_state=RANDOM_STATE)),
])

# ── Hyperparameter grid ──────────────────────────────────
param_grid = {
    "classifier__n_estimators":  [100, 200, 300],
    "classifier__max_depth":     [5, 10, 15, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf":  [1, 2, 4],
}

print(f"Total parameter combinations: {3 * 4 * 3 * 3} = {3 * 4 * 3 * 3}")
print(f"Total fits (with 10-fold CV): {3 * 4 * 3 * 3 * 10}")
print("\n⏳ Running GridSearchCV...")

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

print(f"\n✅ Best CV Accuracy: {grid_search.best_score_:.4f}")
print(f"   Best Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"     {param.replace('classifier__', '')}: {value}")

In [ ]:
# ── Top 10 hyperparameter configurations ─────────────────
results_df = pd.DataFrame(grid_search.cv_results_)
top10 = results_df.nsmallest(10, "rank_test_score")[
    ["rank_test_score", "mean_test_score", "std_test_score",
     "mean_train_score", "mean_fit_time"]
].reset_index(drop=True)

top10.columns = ["Rank", "Mean CV Acc", "Std CV Acc", "Mean Train Acc", "Fit Time (s)"]
top10

---
## 8 · Final Evaluation on Untouched Test Set

In [ ]:
# ── Predict on the held-out test set ─────────────────────
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

test_accuracy = accuracy_score(y_test, y_pred)
test_auc      = roc_auc_score(y_test, y_prob)

print("═" * 55)
print(" FINAL TEST SET EVALUATION")
print("═" * 55)
print(f"  Accuracy : {test_accuracy:.4f}")
print(f"  ROC-AUC  : {test_auc:.4f}")
print("═" * 55)
print()
print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Died", "Survived"],
    cmap="Blues",
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix")

# (b) ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color="#2563eb", lw=2, label=f"AUC = {test_auc:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray", lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.1, color="#2563eb")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Best Random Forest")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance (from the tuned Random Forest) ────
# Extract feature names from the preprocessor
fitted_preprocessor = best_model.named_steps["preprocessor"]

# Numeric features keep their names
num_names = numeric_features

# Categorical features get expanded by OneHotEncoder
cat_encoder = fitted_preprocessor.named_transformers_["cat"].named_steps["encoder"]
cat_names = cat_encoder.get_feature_names_out(categorical_features).tolist()

all_feature_names = num_names + cat_names

# Get importances
importances = best_model.named_steps["classifier"].feature_importances_

feat_imp_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": importances,
}).sort_values("Importance", ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_imp_df["Feature"], feat_imp_df["Importance"], color="#3b82f6")
ax.set_xlabel("Importance (Gini)")
ax.set_title("Feature Importance — Tuned Random Forest")
plt.tight_layout()
plt.show()

---
## 9 · Model Persistence with `joblib`

In [ ]:
# ── Save the best pipeline (preprocessor + classifier) ──
MODEL_PATH = "best_rf_pipeline.joblib"

joblib.dump(best_model, MODEL_PATH)
print(f"✅ Model saved to: {MODEL_PATH}")

# ── Verify: reload and predict ───────────────────────────
loaded_model = joblib.load(MODEL_PATH)
reload_preds = loaded_model.predict(X_test)
assert np.array_equal(y_pred, reload_preds), "Prediction mismatch after reload!"
print(f"✅ Reload verification passed — predictions are identical.")
print(f"   File size: {__import__('os').path.getsize(MODEL_PATH) / 1024:.1f} KB")

---
## 10 · Summary

| Metric | Value |
|--------|-------|
| **Logistic Regression (CV)** | See Section 6 |
| **Decision Tree (CV)** | See Section 6 |
| **Best Random Forest (CV)** | See Section 7 |
| **Test Accuracy** | See Section 8 |
| **Test ROC-AUC** | See Section 8 |

### Key Takeaways

1. **Stratified splitting** preserved the ~38% survival rate in both train and test sets, preventing evaluation bias.
2. **`ColumnTransformer`** ensured that numeric imputation/scaling and categorical encoding were fit on training data only — preventing data leakage.
3. **Logistic Regression** provided a strong linear baseline; the **Decision Tree** showed higher variance across folds.
4. **GridSearchCV** on Random Forest identified the best hyperparameter configuration with regularized tree depth.
5. The final model was **persisted with `joblib`** and verified to produce identical predictions after deserialization.